# 03 — Embeddings transformer (FinBERT y SBERT)

Generamos las representaciones de documentos sobre las que se ejecutará la fase de modelado (UMAP + HDBSCAN) en el siguiente código. Cada earnings call se codifica con dos modelos transformer pre-entrenados:

- **FinBERT** (`ProsusAI/finbert`): BERT financiero entrenado sobre informes 10-K, transcripciones y artículos financieros. Excluimos los tokens especiales del mean pooling porque el [CLS] de FinBERT se ajustó para clasificación de sentimiento; mantenerlo en una agregación de pooling distorsionaría el centroide del documento.
- **SBERT** (`sentence-transformers/all-mpnet-base-v2`): Incluimos todos los tokens en el pooling por ser consistente con su entrenamiento y normalizamos (norma 2) al final.

Para documentos más largos que 512 tokens, aplicamos ventanas deslizantes con solapamiento de 64 tokens y un **mean pooling ponderado por bloque**. El detalle matemático va en la sección 2.

## Configuración inicial

Montamos Google Drive (si estamos en Colab) y nos situamos en la raíz del proyecto, donde están los datos generados en los cuadernos anteriores. El resto del cuaderno usa rutas relativas a partir de ahí.

In [1]:
import os
import sys
from pathlib import Path

# Raíz del proyecto. En Colab los datos están en Drive; en local, en el cwd.
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/TFG")
else:
    BASE_DIR = Path.cwd()

os.chdir(BASE_DIR)
os.makedirs("data/matrices", exist_ok=True)
print(f"Directorio de trabajo: {BASE_DIR}")

Mounted at /content/drive
Entorno:  Colab
BASE_DIR: /content/drive/MyDrive/TFG  (encontrado via: parquet)

Verificando parquet en data/processed/ ...
  OK   Dataset_Transcripciones_Limpio.parquet                103.8 MB  (8,902 filas)
  OK   Dataset_lematizado.parquet                            155.9 MB  (8,893 filas)
  OK   earnings_calls.parquet                                559.2 MB  (8,939 filas)


In [2]:
import os
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [3]:
# Rutas (relativas a la raíz del proyecto)
ruta_processed = "./data/processed"
ruta_matrices  = "./data/matrices"

ruta_dataset_limpio = os.path.join(ruta_processed, "Dataset_Transcripciones_Limpio.parquet")

os.makedirs(ruta_matrices, exist_ok=True)

# Parámetros de inferencia
SEED            = 42      # misma semilla que UMAP 
MAX_TOKENS      = 512     # ventana del transformer
OVERLAP         = 64      # solape entre ventanas consecutivas
CHECKPOINT_FREQ = 50      # checkpoint cada N documentos para ir guardando resultados de (FinBERT y SBERT)

# Modelos en HuggingFace
FINBERT_MODEL = "ProsusAI/finbert"
SBERT_MODEL   = "sentence-transformers/all-mpnet-base-v2"

# Configuración por modelo. Las dos opciones críticas se justifican en la sección 2:
#   - excluir_especiales: si [CLS]/[SEP] entran o no en el mean pooling
#   - normalizar_l2:      si el vector final se proyecta a la hiperesfera unitaria
config_finbert = {
    "hf_name": FINBERT_MODEL,
    "excluir_especiales": True,    # FinBERT: [CLS] entrenado para sentimiento, fuera del pooling
    "normalizar_l2":      False,   # FinBERT no fue optimizado bajo similitud coseno
    "matriz_path":    os.path.join(ruta_matrices, "Matriz_FinBERT.npy"),
    "ids_path":       os.path.join(ruta_matrices, "Matriz_FinBERT_ids.npy"),
    "parquet_path":   os.path.join(ruta_matrices, "Embeddings_FinBERT.parquet"),
    "checkpoint_path":os.path.join(ruta_matrices, "checkpoint_finbert.pkl"),
}
config_sbert = {
    "hf_name": SBERT_MODEL,
    "excluir_especiales": False,   # SBERT: pooling incluye [CLS]/[SEP] (como en su entrenamiento)
    "normalizar_l2":      True,    # SBERT optimizado bajo similitud coseno; por lo que son vectores unitarios
    "matriz_path":    os.path.join(ruta_matrices, "Matriz_SBERT.npy"),
    "ids_path":       os.path.join(ruta_matrices, "Matriz_SBERT_ids.npy"),
    "parquet_path":   os.path.join(ruta_matrices, "Embeddings_SBERT.parquet"),
    "checkpoint_path":os.path.join(ruta_matrices, "checkpoint_sbert.pkl"),
}

## 1. Reproducibilidad de embeddings

Fijamos semillas en `random`, `numpy` y `torch` para reducir variaciones entre ejecuciones. Además, configuramos cuDNN en modo determinista:

```python
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
```
No utilizamos torch.use_deterministic_algorithms(True) porque algunas operaciones internas de transformers generan errores en CUDA cuando se fuerza determinismo completo.

Con esta configuración, los embeddings se mantienen estables entre ejecuciones realizadas sobre la misma GPU.

En pruebas hechas sobre GPUs distintas (por ejemplo T4 y L4) aparecen diferencias numéricas muy pequeñas (~1e-6), aunque no afectan de forma apreciable a métricas posteriores como silhouette o DBCV.

Para evitar inconsistencias entre fases del análisis, las matrices generadas se guardan en data/matrices/ y se reutilizan en los notebooks posteriores.

In [4]:
def fijar_semillas_inferencia(seed=SEED):
    """Congela PRNG y backend cuDNN antes de cualquier op torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return {
        "seed": int(seed),
        "cuda_disponible": bool(torch.cuda.is_available()),
        "cudnn_deterministic": True,
        "cudnn_benchmark": False,
    }


estado_det = fijar_semillas_inferencia()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Dispositivo:           {device}")
print(f"Seed (PRNG + CUDA):    {estado_det['seed']}")
print(f"cuDNN deterministic:   {estado_det['cudnn_deterministic']}")
print(f"cuDNN benchmark off:   {not estado_det['cudnn_benchmark']}")

if not torch.cuda.is_available():
    print()
    print("AVISO: no se ha detectado GPU. La inferencia en CPU tarda ~30× más.")
    print("En Colab: Runtime → Change runtime type → GPU (T4 / L4).")

Dispositivo:           cuda
Seed (PRNG + CUDA):    42
cuDNN deterministic:   True
cuDNN benchmark off:   True


## 2. Ventanas deslizantes y mean pooling ponderado por bloque

Los earnings calls tienen una cantidad de tokens muy por encima del límite de 512 del transformer. Para codificarlos sin truncar empleamos ventanas deslizantes con un esquema de ponderación que garantiza **conservación agregada de masa**: Σ w_i = total_tokens a nivel de bloque.

Parámetros: ventana de `MAX_TOKENS = 512` posiciones totales (510 de contenido efectivo, dejando hueco para los tokens especiales [CLS] y [SEP]) y solape `OVERLAP = 64` tokens. El avance entre ventanas consecutivas es `SALTO = 510 − 64 = 446`.

Los pesos asignados a cada bloque son:

- **Documento mono-ventana** (n_bloques = 1): w_0 = len(bloque_0).
- **Primer bloque** (idx = 0, n_bloques > 1): w_0 = len(bloque_0) (no tiene predecesor).
- **Bloques intermedios** (0 < idx < n_bloques − 1): w_i = SALTO = 446 (tokens nuevos respecto al bloque anterior).
- **Último bloque** (idx = n_bloques − 1, n_bloques > 1): w_{N−1} = len(bloque_{N−1}) − OVERLAP (su tramo izquierdo de 64 tokens ya fue contabilizado por el penúltimo bloque).


En las regiones de solape algunos tokens terminan influyendo más de una vez en el embedding final, mientras que otros aparecen una sola vez. Una agregación completamente exacta a nivel token exigiría almacenar embeddings individuales para todas las posiciones del documento, lo que incrementa mucho el coste en memoria y tiempo de cálculo.

Dado el tamaño del corpus, utilizamos una aproximación basada en bloques solapados, suficiente para mantener estabilidad en los embeddings agregados sin penalizar demasiado el rendimiento computacional.

In [5]:
def obtener_embedding_documento(texto, tokenizer, model, device,
                                excluir_especiales=True, normalizar_l2=False):
    """Embedding de un documento con ventanas deslizantes y mean pooling ponderado.

    Devuelve (vector_final, n_ventanas).
    """
    CONTENT = MAX_TOKENS - 2     # 510 tokens de contenido (sin [CLS] ni [SEP])
    SALTO   = CONTENT - OVERLAP  # 446 tokens nuevos por ventana

    # Tokenizamos sin tokens especiales para tener libertad al cortar los bloques
    tokens = tokenizer(texto, add_special_tokens=False, return_tensors='pt', truncation=False)
    input_ids = tokens['input_ids'][0]
    total_tokens = input_ids.size(0)

    # Construimos bloques con ventana deslizante
    if total_tokens <= CONTENT:
        bloques = [input_ids]
    else:
        bloques = []
        for i in range(0, total_tokens, SALTO):
            bloque = input_ids[i : i + CONTENT]
            bloques.append(bloque)
            if i + CONTENT >= total_tokens:
                break

    embeddings_bloques = []
    pesos_bloques = []
    n_bloques = len(bloques)

    with torch.no_grad():
        for idx, bloque in enumerate(bloques):
            # Añadimos [CLS] y [SEP] al bloque antes de pasarlo al modelo
            bloque_completo = torch.cat([
                torch.tensor([tokenizer.cls_token_id]),
                bloque,
                torch.tensor([tokenizer.sep_token_id]),
            ]).unsqueeze(0).to(device)

            attention_mask = torch.ones_like(bloque_completo).to(device)
            outputs = model(bloque_completo, attention_mask=attention_mask)

            # Mean pooling con exclusión condicional de [CLS]/[SEP]
            if excluir_especiales:
                hidden = outputs.last_hidden_state[0, 1:-1, :]
            else:
                hidden = outputs.last_hidden_state[0, :, :]

            mean_pooled = torch.mean(hidden, dim=0)
            embeddings_bloques.append(mean_pooled.cpu().numpy())

            # Peso del bloque — esquema de conservación de masa Σ w_i = total_tokens
            if n_bloques == 1 or idx == 0:
                pesos_bloques.append(len(bloque))
            elif idx == n_bloques - 1:
                pesos_bloques.append(max(len(bloque) - OVERLAP, 0))
            else:
                pesos_bloques.append(SALTO)

    # Centroide del documento ponderado por bloque
    vec_final = np.average(embeddings_bloques, axis=0, weights=pesos_bloques)

    # Normalización L2 condicional (SBERT sí, FinBERT no)
    if normalizar_l2:
        norma = np.linalg.norm(vec_final)
        if norma > 0:
            vec_final = vec_final / norma

    return vec_final, n_bloques

## 3. Generación de los embeddings

Cargamos los documentos válidos (`texto_valido == 1`) y recorremos cada modelo. Como la inferencia de los dos modelos tarda más de una hora en total, guardamos un checkpoint cada `CHECKPOINT_FREQ` documentos; si la sesión de Colab se corta, al reejecutar la celda el bucle continúa desde el último checkpoint.

Por cada modelo se guardan tres ficheros:

- `Matriz_<MODELO>.npy` — la matriz de embeddings (`n_documentos × dim`).
- `Matriz_<MODELO>_ids.npy` — los `tupla_id` en el mismo orden que las filas, para poder comprobar después que cada embedding va con su documento.
- `Embeddings_<MODELO>.parquet` — la misma matriz en formato tabla, por comodidad.

In [6]:
df = pd.read_parquet(ruta_dataset_limpio)
df_validos = df[df['texto_valido'] == 1].copy()
textos    = df_validos['presentation'].tolist()
tuplas_id = df_validos['tupla_id'].tolist()

Documentos a codificar: 8,893
Ejemplo de tupla_id:    'LEN_2019Q1'


In [7]:
def generar_embeddings(cfg, textos, tuplas_id, device):
    """Genera los embeddings de un modelo y guarda matriz, ids y parquet.

    Usa un checkpoint .pkl para poder reanudar si Colab se desconecta.
    """
    print(f"\nMODELO: {cfg['hf_name']}  "
          f"(excluir_especiales={cfg['excluir_especiales']}, "
          f"normalizar_l2={cfg['normalizar_l2']})")

    tokenizer = AutoTokenizer.from_pretrained(cfg['hf_name'])
    model = AutoModel.from_pretrained(cfg['hf_name']).to(device)
    model.eval()

    # Si hay checkpoint previo, retomamos desde donde se quedó.
    checkpoint_path = Path(cfg['checkpoint_path'])
    lista_embeddings, lista_n_ventanas = [], []
    if checkpoint_path.exists():
        with checkpoint_path.open('rb') as f:
            lista_embeddings, lista_n_ventanas = pickle.load(f)
        print(f"  Reanudando desde {len(lista_embeddings)} documentos ya procesados.")

    inicio_idx = len(lista_embeddings)
    for i, texto in enumerate(tqdm(textos[inicio_idx:], initial=inicio_idx, total=len(textos))):
        emb, n_vent = obtener_embedding_documento(
            texto, tokenizer, model, device,
            excluir_especiales=cfg['excluir_especiales'],
            normalizar_l2=cfg['normalizar_l2'],
        )
        lista_embeddings.append(emb)
        lista_n_ventanas.append(n_vent)

        if (inicio_idx + i + 1) % CHECKPOINT_FREQ == 0:
            with checkpoint_path.open('wb') as f:
                pickle.dump((lista_embeddings, lista_n_ventanas), f)

    # Guardamos los tres ficheros de salida.
    matriz = np.vstack(lista_embeddings)
    np.save(cfg['matriz_path'], matriz)
    np.save(cfg['ids_path'], np.asarray(tuplas_id, dtype=object))
    pd.DataFrame({'tupla_id': tuplas_id, 'embedding': list(matriz)}) \
        .to_parquet(cfg['parquet_path'], index=False)

    # Ya está todo guardado: borramos el checkpoint.
    if checkpoint_path.exists():
        checkpoint_path.unlink()

    print(f"  Matriz {matriz.shape} guardada en {cfg['matriz_path']}")
    return matriz, lista_n_ventanas

In [8]:
matriz_finbert, n_vent_finbert = generar_embeddings(config_finbert, textos, tuplas_id, device)

normas_fb = np.linalg.norm(matriz_finbert, axis=1)
n_vent_validas_fb = [n for n in n_vent_finbert if n > 0]
print()
print(f"FinBERT — n_documentos:        {matriz_finbert.shape[0]:,}")
print(f"FinBERT — dim_embedding:       {matriz_finbert.shape[1]}")
print(f"FinBERT — norma L2 (media):    {normas_fb.mean():.4f}")
print(f"FinBERT — norma L2 (std):      {normas_fb.std():.4f}")
print(f"FinBERT — ventanas/doc (media):{np.mean(n_vent_validas_fb):.2f}")


  MODELO: ProsusAI/finbert
  excluir_especiales=True  |  normalizar_l2=False


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Reanudando desde checkpoint: 3800 documentos ya procesados.
  Procesando 5093 documentos restantes...



100%|██████████| 8893/8893 [34:01<00:00,  2.49it/s]



  Matriz: (8893, 768)    →  ./data/matrices/Matriz_FinBERT.npy
  IDs   : 8893      →  ./data/matrices/Matriz_FinBERT_ids.npy
  Parquet                       →  ./data/matrices/Embeddings_FinBERT.parquet

FinBERT — n_documentos:        8,893
FinBERT — dim_embedding:       768
FinBERT — norma L2 (media):    9.8144
FinBERT — norma L2 (std):      0.4100
FinBERT — ventanas/doc (media):11.10


In [9]:
matriz_sbert, n_vent_sbert = generar_embeddings(config_sbert, textos, tuplas_id, device)

normas_sb = np.linalg.norm(matriz_sbert, axis=1)
n_vent_validas_sb = [n for n in n_vent_sbert if n > 0]
print()
print(f"SBERT — n_documentos:          {matriz_sbert.shape[0]:,}")
print(f"SBERT — dim_embedding:         {matriz_sbert.shape[1]}")
print(f"SBERT — norma L2 (media):      {normas_sb.mean():.4f}  (≈ 1 por normalización L2)")
print(f"SBERT — norma L2 (std):        {normas_sb.std():.4f}")
print(f"SBERT — ventanas/doc (media):  {np.mean(n_vent_validas_sb):.2f}")


  MODELO: sentence-transformers/all-mpnet-base-v2
  excluir_especiales=False  |  normalizar_l2=True


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Sin checkpoint previo, empezando desde cero.
  Procesando 8893 documentos restantes...



100%|██████████| 8893/8893 [1:13:21<00:00,  2.02it/s]



  Matriz: (8893, 768)    →  ./data/matrices/Matriz_SBERT.npy
  IDs   : 8893      →  ./data/matrices/Matriz_SBERT_ids.npy
  Parquet                       →  ./data/matrices/Embeddings_SBERT.parquet

SBERT — n_documentos:          8,893
SBERT — dim_embedding:         768
SBERT — norma L2 (media):      1.0000  (≈ 1 por normalización L2)
SBERT — norma L2 (std):        0.0000
SBERT — ventanas/doc (media):  11.10
